In [0]:
from pyspark.sql import functions as F

customers   = spark.table("bronze_customers")
orders      = spark.table("bronze_orders")
order_items = spark.table("bronze_order_items")
products    = spark.table("bronze_products")
payments    = spark.table("bronze_payments")

customers_clean = customers.dropDuplicates(["customer_id"]).dropna(subset=["customer_id"])

orders_clean = (
    orders
    .dropDuplicates(["order_id"])
    .dropna(subset=["order_id", "customer_id"])
    .withColumn("order_purchase_timestamp", F.to_timestamp("order_purchase_timestamp"))
)

order_items_clean = (
    order_items
    .dropDuplicates(["order_id", "order_item_id"])
    .dropna(subset=["order_id", "product_id"])
)

products_clean = (
    products
    .dropDuplicates(["product_id"])
    .withColumn(
        "product_category_name",
        F.coalesce(F.col("product_category_name"), F.lit("unknown"))
    )
)

payments_clean = payments.dropDuplicates(["order_id", "payment_sequential"]).dropna(subset=["order_id"])

print("Rows after cleaning:")
print("customers:", customers_clean.count())
print("orders:", orders_clean.count())
print("order_items:", order_items_clean.count())
print("products:", products_clean.count())
print("payments:", payments_clean.count())

silver_order_details = (
    orders_clean.alias("o")
    .join(customers_clean.alias("c"), "customer_id", "inner")
    .join(order_items_clean.alias("oi"), "order_id", "inner")
    .join(products_clean.alias("p"), "product_id", "inner")
    .join(payments_clean.alias("pay"), "order_id", "inner")
    .select(
        F.col("o.order_id"),
        F.col("c.customer_id"),
        F.col("c.customer_city"),
        F.col("c.customer_state"),
        F.col("p.product_id"),
        F.col("p.product_category_name").alias("product_category"),
        F.col("oi.price"),
        F.col("oi.freight_value"),
        F.col("pay.payment_type"),
        F.col("pay.payment_value"),
        F.col("o.order_purchase_timestamp").alias("purchase_timestamp"),
        F.col("o.order_status"),
    )
    .dropDuplicates()
)

silver_order_details.write.format("delta").mode("overwrite").saveAsTable("silver_order_details")

print("silver_order_details rows:", silver_order_details.count())
silver_order_details.show(5)

Rows after cleaning:
customers: 99441
orders: 99441
order_items: 112650
products: 32951
payments: 103886
silver_order_details rows: 106386
+--------------------+--------------------+---------------+--------------+--------------------+--------------------+------+-------------+------------+-------------+-------------------+------------+
|            order_id|         customer_id|  customer_city|customer_state|          product_id|    product_category| price|freight_value|payment_type|payment_value| purchase_timestamp|order_status|
+--------------------+--------------------+---------------+--------------+--------------------+--------------------+------+-------------+------------+-------------+-------------------+------------+
|136cce7faa42fdb2c...|ed0271e0b7da060a3...|     santa rosa|            RS|a1804276d9941ac07...|             unknown|  49.9|        16.05| credit_card|        65.95|2017-04-11 12:22:08|    invoiced|
|e69bfb5eb88e0ed6a...|31ad1d1b63eb99624...|       sorocaba|          